In [2]:
import torch
from onnx2torch import convert

onnx_model_path = "../model.onnx"
student = convert(onnx_model_path)

# student = torch.nn.Sequential(
#     torch.nn.Conv2d(81, 3, kernel_size=3, padding=1),
#     student,
# )

In [11]:
for name, param in student.named_parameters():
    print(name)

conv1/Conv.weight
conv1/Conv.bias
layer1/layer1/0/bn1/BatchNormalization.weight
layer1/layer1/0/bn1/BatchNormalization.bias
layer1/layer1/0/conv1/Conv.weight
layer1/layer1/0/conv1/Conv.bias
layer1/layer1/0/conv2/Conv.weight
layer1/layer1/0/conv2/Conv.bias
layer1/layer1/0/downsample/downsample/0/Conv.weight
layer1/layer1/0/downsample/downsample/0/Conv.bias
layer1/layer1/1/bn1/BatchNormalization.weight
layer1/layer1/1/bn1/BatchNormalization.bias
layer1/layer1/1/conv1/Conv.weight
layer1/layer1/1/conv1/Conv.bias
layer1/layer1/1/conv2/Conv.weight
layer1/layer1/1/conv2/Conv.bias
layer1/layer1/2/bn1/BatchNormalization.weight
layer1/layer1/2/bn1/BatchNormalization.bias
layer1/layer1/2/conv1/Conv.weight
layer1/layer1/2/conv1/Conv.bias
layer1/layer1/2/conv2/Conv.weight
layer1/layer1/2/conv2/Conv.bias
layer2/layer2/0/bn1/BatchNormalization.weight
layer2/layer2/0/bn1/BatchNormalization.bias
layer2/layer2/0/conv1/Conv.weight
layer2/layer2/0/conv1/Conv.bias
layer2/layer2/0/conv2/Conv.weight
layer2/l

In [ ]:
student_2 = convert(onnx_model_path)
student_2 = torch.nn.Sequential(
    torch.nn.Conv2d(81, 3, kernel_size=3, padding=1),
    student_2,
) 
student_2.load_state_dict(torch.load("student.pth", map_location='cpu'))

<All keys matched successfully>

In [16]:
original_weights = student.state_dict()
student_2_weights = student_2.state_dict()
delta = {}
for key in original_weights:
    if "conv" in key or "fc" in key:
        delta[key] = torch.mean((student_2_weights[key] - original_weights[key]) / original_weights[key].abs())

In [18]:
layers = {}
for key in delta:
    layer = key.split(".")[1].split("/")[0]
    old = layers.get(layer, [])
    old.append(delta[key].item())
    layers[layer] = old

In [21]:
layers = {k: sum(v)/len(v) for k, v in layers.items()}

In [ ]:
layers 

{'conv1': 239.90256494283676,
 'layer1': -88.60885560760896,
 'layer2': 7.177278610387405,
 'layer3': 4.658421500077626,
 'layer4': 4.014495396986604,
 'fc': 1569.049602508545}